In [57]:
import pandas as pd
import altair as alt

In [3]:
all_df = pd.read_csv("./dataframes/all_evaluated_responses_df.csv")
all_df = all_df.dropna(subset=['agent_response'])

In [43]:
test_items = all_df[['item_id', 'test_type']].drop_duplicates().reset_index()
test_items = test_items[['test_type']].value_counts().reset_index()
test_items = test_items.rename(columns={'count': 'item_count'})
test_items['item_count'] = test_items['item_count'] * 10

In [75]:
# get propotion of valid responses 
# one assumption here is that every model only answers a question 10 items
valid_response_counts = all_df[['test_type', 'agent_type']].value_counts().reset_index()
valid_response_counts = pd.merge(valid_response_counts, test_items, on='test_type')
valid_response_counts['proportion_valid'] = valid_response_counts['count'] / valid_response_counts['item_count']
valid_response_counts.to_csv("./dataframes/machine_valid_response.csv")

In [49]:
test_order = [
    "ggr",
    "vlat",
    'calvi',
    "holf",
    'holf2',
    'chartqa-test-continuous-human',
]

model_configs = [
    {'model': "Salesforce/blip2-flan-t5-xl", 'top_p': 'p06', 'temp': 't1', 'color': '#5ba3cf', 'accent_color': '#8cbedd'},
    {'model': "Salesforce/blip2-flan-t5-xxl", 'top_p': 'p1', 'temp': 't10', 'color': '#4c78a8', 'accent_color': '#81a0c2'},
    {'model': "llava-hf/llava-1.5-7b-hf", 'top_p': 'p04', 'temp': 't1', 'color': '#f9b574', 'accent_color': '#facb9d'},
    {'model': "llava-hf/llava-1.5-13b-hf", 'top_p': 'p1', 'temp': 't04', 'color': '#f58518', 'accent_color': '#f79d46'},
    {'model': "liuhaotian/llava-v1.6-34b", 'top_p': 'p1', 'temp': 't04', 'color': '#BF9000', 'accent_color': '#F1C232'},
    {'model': "google/pix2struct-chartqa-base", 'top_p': 'p08', 'temp': 't1', 'color': '#ba94c4', 'accent_color': '#c7b8d9'},
    {'model': "google/matcha-chartqa", 'top_p': 'p04', 'temp': 't1', 'color': '#8b6db2', 'accent_color': '#a28ac1'},
    {'model': "GPT-4V", 'top_p': 'p1', 'temp': 't02', 'color': '#c93739', 'accent_color': '#C6765E'},
]

In [74]:
charts = []
for model_config in model_configs:
    main_color = model_config['color']
    model = model_config['model']
    df = valid_response_counts[
        valid_response_counts['agent_type'] == model
    ]

    if model == 'Salesforce/blip2-flan-t5-xl':
        yaxis = alt.Axis()
    else:
        yaxis = None
    
    chart = alt.Chart(df).mark_bar(color=main_color).encode(
        x=alt.X('test_type', title=None, scale=alt.Scale(domain=test_order)),
        y=alt.Y('proportion_valid', axis=yaxis, title=None, scale=alt.Scale(domain=[0, 1]) ),
        color=alt.value(main_color),
    ).properties(width=80, height=100)
    charts.append(chart)

prop_valid_responses = alt.hconcat(*charts)
prop_valid_responses.save("./figures/machine_valid_response.pdf")
prop_valid_responses

alt.HConcatChart(...)

## Total propotion of valid responses

In [100]:
full_valid_counts = valid_response_counts.groupby(['agent_type']).agg(
    item_count=('item_count', 'sum'),
    count=('count', 'sum')
).reset_index()
full_valid_counts['proportion'] = full_valid_counts['count'] / full_valid_counts['item_count']

In [101]:
full_valid_counts.sort_values(by='proportion')

,agent_type,item_count,count,proportion
6,liuhaotian/llava-v1.6-34b,8510,2735,0.321387
7,llava-hf/llava-1.5-13b-hf,8510,3202,0.376263
8,llava-hf/llava-1.5-7b-hf,8510,3964,0.465805
0,GPT-4V,8510,5288,0.621387
5,google/pix2struct-chartqa-base,8510,6385,0.750294
3,Salesforce/blip2-flan-t5-xxl,8510,6926,0.813866
2,Salesforce/blip2-flan-t5-xl,8510,7040,0.827262
4,google/matcha-chartqa,8510,7082,0.832197
1,Human,8510,95565,11.229730


### Average prop. valid across tests

In [104]:
valid_response_counts.groupby(['agent_type'])['proportion_valid'].mean().reset_index().sort_values(by='proportion_valid')

,agent_type,proportion_valid
6,liuhaotian/llava-v1.6-34b,0.510036
7,llava-hf/llava-1.5-13b-hf,0.563341
5,google/pix2struct-chartqa-base,0.611932
8,llava-hf/llava-1.5-7b-hf,0.614426
4,google/matcha-chartqa,0.646338
0,GPT-4V,0.721774
3,Salesforce/blip2-flan-t5-xxl,0.803084
2,Salesforce/blip2-flan-t5-xl,0.818405
1,Human,40.812827
